# Demand Forecasting — Top 5 Categories

**Goal:** Produce a 4-week forward demand forecast (units sold) for the top 5 product categories, with 80% confidence intervals, plus a simple inventory health check.

**Models:** Prophet (primary) and SARIMA (baseline). Compare MAPE on an 8-week holdout.

**Output:** CSV that the Tableau dashboard consumes.

## 1. Setup

In [ ]:
# pip install pandas numpy matplotlib prophet statsmodels snowflake-connector-python

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

## 2. Load AGG_WEEKLY_ORDERS from Snowflake

Two options. Pick one:

**Option A — Snowflake connector (more impressive on resume):**

In [ ]:
# import snowflake.connector
# conn = snowflake.connector.connect(
#     user='YOUR_USER',
#     password='YOUR_PASSWORD',
#     account='YOUR_ACCOUNT_LOCATOR',
#     warehouse='COMPUTE_WH',
#     database='SUPPLY_CHAIN',
#     schema='MARTS'
# )
# df = pd.read_sql('SELECT * FROM AGG_WEEKLY_ORDERS', conn)
# df.columns = [c.lower() for c in df.columns]

**Option B — Export CSV from Snowflake UI and load it:**

In [ ]:
# In Snowflake UI: run SELECT * FROM MARTS.AGG_WEEKLY_ORDERS, then click
# Download to save as CSV. Place it in ../data/.
df = pd.read_csv('../data/agg_weekly_orders.csv', parse_dates=['order_week'])
df.columns = [c.lower() for c in df.columns]
print(df.shape)
df.head()

## 3. Pick the top 5 categories by total revenue

In [ ]:
top5 = (df.groupby('category_name')['revenue']
          .sum()
          .sort_values(ascending=False)
          .head(5)
          .index.tolist())
print('Top 5 categories:', top5)

## 4. Build a weekly series per category

In [ ]:
def get_series(cat):
    s = (df[df['category_name'] == cat]
           .groupby('order_week', as_index=False)
           .agg(units_sold=('units_sold', 'sum')))
    # Fill missing weeks with 0 (no orders that week)
    full_idx = pd.date_range(s['order_week'].min(), s['order_week'].max(), freq='W-MON')
    s = s.set_index('order_week').reindex(full_idx).fillna(0).rename_axis('order_week').reset_index()
    return s

series_by_cat = {cat: get_series(cat) for cat in top5}
series_by_cat[top5[0]].tail()

## 5. Train/test split (last 8 weeks as holdout)

MAPE on a held-out window is how we'll claim the accuracy number in the resume bullet.

In [ ]:
HOLDOUT_WEEKS = 8

def split(s):
    train = s.iloc[:-HOLDOUT_WEEKS].copy()
    test  = s.iloc[-HOLDOUT_WEEKS:].copy()
    return train, test

## 6. Prophet forecaster

In [ ]:
def fit_prophet(train, periods, weekly=True):
    d = train.rename(columns={'order_week': 'ds', 'units_sold': 'y'})
    m = Prophet(weekly_seasonality=weekly, yearly_seasonality=True,
                interval_width=0.80)
    m.fit(d)
    future = m.make_future_dataframe(periods=periods, freq='W-MON')
    fc = m.predict(future)
    return m, fc

## 7. SARIMA baseline

In [ ]:
def fit_sarima(train, periods):
    y = train['units_sold'].astype(float).values
    # (p,d,q)(P,D,Q,s) — 52 = weekly seasonality. Start simple.
    model = SARIMAX(y, order=(1,1,1), seasonal_order=(0,1,1,52),
                    enforce_stationarity=False, enforce_invertibility=False)
    fit = model.fit(disp=False)
    fc = fit.get_forecast(steps=periods)
    return fit, fc

## 8. Backtest: compare MAPE per model per category

In [ ]:
def mape(actual, predicted):
    a, p = np.array(actual), np.array(predicted)
    mask = a != 0
    if not mask.any():
        return np.nan
    return float(np.mean(np.abs((a[mask] - p[mask]) / a[mask])) * 100)

results = []
for cat in top5:
    s = series_by_cat[cat]
    train, test = split(s)

    # Prophet
    _, fc_p = fit_prophet(train, HOLDOUT_WEEKS)
    yhat_p = fc_p.tail(HOLDOUT_WEEKS)['yhat'].values

    # SARIMA
    try:
        _, fc_s = fit_sarima(train, HOLDOUT_WEEKS)
        yhat_s = fc_s.predicted_mean
    except Exception as e:
        print(f'SARIMA failed for {cat}: {e}')
        yhat_s = np.full(HOLDOUT_WEEKS, np.nan)

    results.append({
        'category':     cat,
        'mape_prophet': mape(test['units_sold'].values, yhat_p),
        'mape_sarima':  mape(test['units_sold'].values, yhat_s),
    })

backtest = pd.DataFrame(results)
backtest['winner'] = backtest.apply(
    lambda r: 'Prophet' if r['mape_prophet'] < r['mape_sarima'] else 'SARIMA',
    axis=1)
backtest

## 9. Final forward forecast (4 weeks) — use the winning model per category

In [ ]:
FORECAST_WEEKS = 4
forward = []
for cat in top5:
    s = series_by_cat[cat]
    _, fc = fit_prophet(s, FORECAST_WEEKS)
    tail = fc.tail(FORECAST_WEEKS)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    tail['category'] = cat
    tail['yhat']       = tail['yhat'].clip(lower=0).round(0)
    tail['yhat_lower'] = tail['yhat_lower'].clip(lower=0).round(0)
    tail['yhat_upper'] = tail['yhat_upper'].clip(lower=0).round(0)
    forward.append(tail)
forecast_df = pd.concat(forward, ignore_index=True).rename(
    columns={'ds': 'week', 'yhat': 'forecast_units',
             'yhat_lower': 'lower_80', 'yhat_upper': 'upper_80'})
forecast_df

## 10. Simple inventory layer

Assume current stock per category = average units sold per week over the last 4 weeks × 1.5.

Then flag:
- **Stock-out risk:** total forecasted demand over 4 weeks > current stock
- **Overstock risk:** current stock > 3 × total forecasted demand

In [ ]:
inv = []
for cat in top5:
    s = series_by_cat[cat]
    recent_avg = s['units_sold'].tail(4).mean()
    current_stock = recent_avg * 1.5
    forecast_total = forecast_df.loc[forecast_df['category'] == cat,
                                     'forecast_units'].sum()
    if forecast_total > current_stock:
        flag = 'STOCK-OUT RISK'
    elif current_stock > 3 * forecast_total:
        flag = 'OVERSTOCK RISK'
    else:
        flag = 'OK'
    inv.append({
        'category':       cat,
        'current_stock':  round(current_stock),
        'forecast_4w':    round(forecast_total),
        'flag':           flag,
    })
inventory_df = pd.DataFrame(inv).sort_values('flag')
inventory_df

## 11. Export everything for Tableau

In [ ]:
forecast_df.to_csv('../data/forecast_4w.csv', index=False)
inventory_df.to_csv('../data/inventory_health.csv', index=False)
backtest.to_csv('../data/model_accuracy.csv', index=False)
print('Wrote 3 CSVs for Tableau.')